# Universal Anomaly Detector

## 1. Behavioral Analysis (Generalization)
### Defining a Multi-Domain Security Signature
* **Focus:** Merging Tuesday, Wednesday, and Thursday into one "Universal" brain.
* **Logic:** Learning the shared DNA of all attack types.

In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

df_tue = pd.read_csv('../data/Tuesday-WorkingHours.pcap_ISCX.csv', nrows=50000)
df_wed = pd.read_csv('../data/Wednesday-workingHours.pcap_ISCX.csv', nrows=50000)
df_thu = pd.read_csv('../data/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv', nrows=50000)

df_all = pd.concat([df_tue, df_wed, df_thu], ignore_index=True)

df_all.columns = df_all.columns.str.strip()
df_all['Label'] = df_all['Label'].str.strip()
df_all.replace([np.inf, -np.inf], np.nan, inplace=True)
df_all.dropna(inplace=True)

df_all['IsAnomaly'] = df_all['Label'].apply(lambda x: 0 if x == 'BENIGN' else 1)

print("Balanced Dataset Created!")
print(df_all['IsAnomaly'].value_counts())

Balanced Dataset Created!
IsAnomaly
0    140914
1      8935
Name: count, dtype: int64


## 2. Retraining with the "Clean" Data
### Global Sanitization and Shuffle
* **Action:** Concatenating 150k balanced samples and shuffling for high variance.
* **Result:** Final validated model for the Master's Thesis.

In [2]:
strict_drop_list = ['Label', 'IsAnomaly', 'Flow ID', 'Source IP', 'Destination IP', 
                    'Timestamp', 'Source Port', 'Destination Port']

X = df_all.drop(columns=[c for c in strict_drop_list if c in df_all.columns])
y = df_all['IsAnomaly']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=True)

## 3. Feature Importance Analysis
### The "Universal Fingerprint" Discovery
* **Top Clues:** Identifying why Packet Size beats Timing in a general network.

In [3]:
universal_model = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)

universal_model.fit(X_train, y_train)

y_pred = universal_model.predict(X_test)
print("--- UNIVERSAL MODEL PERFORMANCE ---")
print(classification_report(y_test, y_pred))

--- UNIVERSAL MODEL PERFORMANCE ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     28119
           1       1.00      0.99      1.00      1851

    accuracy                           1.00     29970
   macro avg       1.00      1.00      1.00     29970
weighted avg       1.00      1.00      1.00     29970



In [4]:
importances = universal_model.feature_importances_
feature_info = pd.DataFrame({'Feature': X.columns, 'Importance': importances})
feature_info = feature_info.sort_values(by='Importance', ascending=False)

print("Top 5 Features of the Universal Model:")
print(feature_info.head(5))

Top 5 Features of the Universal Model:
                   Feature  Importance
52    Avg Fwd Segment Size    0.056645
5    Fwd Packet Length Max    0.049751
40       Packet Length Std    0.044062
23             Fwd IAT Min    0.038028
7   Fwd Packet Length Mean    0.035305


In [6]:
import joblib

joblib.dump(universal_model, '../models/universal_anomaly_detector.pkl')

print("Universal Model saved successfully to /models!")

Universal Model saved successfully to /models!
